# Web3 Python 100本ノック：第1章
## §1-2：USDT残高の取得

前回の§1-1では「ETH」の残高を取得しました。ETHはイーサリアムネットワークの基軸通貨であるため、非常に簡単に取得できました。
しかし、USDTやUSDCのような「ERC20トークン」は、ブロックチェーン上にデプロイされた**スマートコントラクト**というプログラムによって管理されています。

このノートでは、Pythonからスマートコントラクトの関数を呼び出し、特定のトークン残高を取得する方法を学びます。


> **注意**: 各セルを順番に実行してください。セルの実行には `Shift + Enter` を押します。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsukumo-999/web3-knock100-first-half/blob/master/s1-2_get_usdt_wallet.ipynb)

## 準備

Colabでこのノートブックを実行する場合は、まず以下のセルを実行して `web3` ライブラリをインストールしてください。

In [ ]:
# Colabで使用する場合、最初にweb3ライブラリをインストールする（!pip install web3==7.16.0を実行）
# ※ ローカル環境では既にインストール済みの場合はコメントアウト、
# このセルを実行しても問題ありませんが、他の環境ではweb3のバージョンが異なる場合があるため、
# バージョンを指定してインストールすることを推奨します。
!pip install web3==7.16.0

## 最小限のABIを定義する
スマートコントラクトと通信するには、「ABI（Application Binary Interface）」と呼ばれる設計図が必要です。

本来、ABIは非常に長いJSONデータ(数kBから数十kB)ですが、今回は「残高を取得する（`balanceOf`）」という1つの機能しか使わないため、<br>
**必要な部分だけをPythonのリスト形式で定義**する方法を使います。


### 1. 接続設定

In [1]:
from web3 import Web3

RPC_URL = "https://eth.drpc.org"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'headers': headers, 'timeout': 5}))

try:
    latest_block = w3.eth.block_number
    print(f"接続成功！ (最新ブロック: {latest_block})")
except Exception as e:
    print(f"接続失敗: {e}")

print("-" * 30)

接続成功！ (最新ブロック: 25658801)
------------------------------


### 2. ERC20トークンの情報設定
ここでは世界最大のステーブルコインであるTether: USDTを使用

In [2]:
USDT_ADDRESS = "0xdAC17F958D2ee523a2206206994597C13D831ec7" # メインネットのUSDTコントラクトアドレス

# 最小限のABI定義（balanceOf関数とdecimals関数だけを定義）
# これにより、スマートコントラクトがどのような関数を持っているかをweb3.pyに教えます
erc20_abi = [
    {
        "constant": True,
        "inputs": [{"name": "_owner", "type": "address"}],
        "name": "balanceOf",
        "outputs": [{"name": "balance", "type": "uint256"}],
        "type": "function"
    },
    {
        "constant": True,
        "inputs": [],
        "name": "decimals",
        "outputs": [{"name": "", "type": "uint8"}],
        "type": "function"
    }
]


### 3. コントラクトオブジェクトの作成
宛先アドレスを「チェックサムアドレス（大文字小文字が混ざった正しい形式）」に変換

In [3]:
contract_address = w3.to_checksum_address(USDT_ADDRESS)
usdt_contract = w3.eth.contract(address=contract_address, abi=erc20_abi)

### 4. 残高を取得したいウォレットアドレス
例としてBinance取引所の巨大ウォレットを指定しています。


In [4]:
target_address = w3.to_checksum_address("0x28C6c06298d514Db089934071355E5743bf21d60")

### 5. コントラクトの関数を呼び出す
usdt_contract.functions.関数名().call()


In [5]:

raw_balance = usdt_contract.functions.balanceOf(target_address).call()
decimals = usdt_contract.functions.decimals().call()


### 6. 人間が読める単位に変換
ETHは10^18が基本ですが、USDTは「10^6」で1ドルになるという特殊な仕様（decimals）を持っています。

In [ ]:
formatted_balance = raw_balance / (10 ** decimals)

print(f"Target Address: {target_address}")
print(f"USDT Balance  : {formatted_balance:,.2f} USDT")
print(f"(Raw value: {raw_balance}, Decimals: {decimals})")

Target Address: 0x28C6c06298d514Db089934071355E5743bf21d60
USDT Balance  : 678,384,347.01 USDT
(Raw value: 678384347007267, Decimals: 6)


### 7. USDT to JPY
USDTでも読めますが、わかりずらい為、USDT=USD(米ドル)として日本円に読み替えてみます。

In [ ]:
import requests # 為替の取得のためAPIを叩くために使用します

# 7. 日本円（JPY）への換算
# CoinGeckoの無料APIを使用して、現在のUSDTの日本円価格（レート）を取得
print("-" * 30)
print("現在のレートで日本円に換算中...")

try:
    # tether(USDT)のJPY(日本円)価格を要求
    api_url = "https://api.coingecko.com/api/v3/simple/price?ids=tether&vs_currencies=jpy"
    response = requests.get(api_url)
    data = response.json()
    
    # JSONデータからレートを抽出
    usdt_to_jpy_rate = data["tether"]["jpy"]

    # 残高(USDT) × レート(円/USDT)
    balance_jpy = formatted_balance * usdt_to_jpy_rate

    print(f"現在のレート: 1 USDT = {usdt_to_jpy_rate} 円")
    print(f"日本円換算  : 約 {balance_jpy:,.0f} 円") # 3桁区切り、小数点以下なしで表示

except Exception as e:
    print(f"レートの取得に失敗しました: {e}")

------------------------------
🔄 現在のレートで日本円に換算中...
現在のレート: 1 USDT = 157.29 円
日本円換算  : 約 106,703,073,941 円
